<center>
<img src="https://supportvectors.ai/logo-poster-transparent.png" width=400px style="opacity:0.8">
</center>

# Knowledge Extraction with the LLM

Extraction is a **two-stage OpenIE pipeline** — the LLM is never asked to "generate schemas":

1. **Named Entity Recognition** — pull the salient entities out of the passage.
2. **RDF triple extraction** — build `(subject, predicate, object)` triples that reuse those entities.

A lightweight **entity-typing** pass then labels each entity, and the abstract **schemas** `(head_type, relation, tail_type)` are *derived* from the typed facts afterward.

The configured model is `openai/gpt-oss-20b` on the bootcamp endpoint.

In [1]:
%run supportvectors-common.ipynb



<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors. All rights reserved. <blockquote>This notebook is the intellectual property of SupportVectors, and part of its training material. 
Only the participants in SupportVectors workshops are allowed to study the notebooks for educational purposes currently, but is prohibited from copying or using it for any other purposes without written permission.

<b> These notebooks are chapters and sections from Asif Qamar's textbook that he is writing on Data Science. So we request you to not circulate the material to others.</b>
 </blockquote>
 <hr/>
</div>



In [2]:

from memg_concepts.config import DEFAULT_PDF
from memg_concepts.document import build_passages
from memg_concepts.extraction import (
    derive_schema_triples,
    extract_from_passage,
    extract_named_entities,
    extract_open_triples,
    to_fact_triples,
    type_entities,
)

passages = build_passages(DEFAULT_PDF, max_chunks=4)
sample = passages[2]
print(sample.text[:400])

translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data. ∗Equal contribution


In [3]:
# Stage 1 + 2: NER, then RDF triples that reuse the named entities.
entities = extract_named_entities(sample)
print('Named entities:', entities)

triples = extract_open_triples(sample, entities)
print('OpenIE triples:', triples[:8])

# Entity typing turns raw entities into a type lookup.
print('Entity types:', type_entities(entities, sample))

Named entities: ['Jakob', 'Ashish', 'Illia', 'Noam', 'Transformer', 'BLEU', 'English', 'English constituency parsing', 'RNNs', '3.5 days', 'eight GPUs']
OpenIE triples: [['Our model', 'established BLEU score', '41.8'], ['Our model', 'trained for', '3.5 days'], ['Our model', 'trained on', 'eight GPUs'], ['Transformer', 'generalizes to', 'other tasks'], ['Transformer', 'applied to', 'English constituency parsing'], ['Jakob', 'proposed replacing', 'RNNs with self-attention'], ['Jakob', 'started', 'evaluation effort'], ['Ashish', 'designed', 'first Transformer models']]
Entity types: {'jakob': 'person', 'ashish': 'person', 'illia': 'person', 'noam': 'person', 'transformer': 'model', 'bleu': 'metric', 'english': 'concept', 'english constituency parsing': 'task', 'rnns': 'model', '3.5 days': 'date', 'eight gpus': 'tool'}


In [4]:
# The full pipeline: NER -> triples -> typing -> typed facts.
extraction = extract_from_passage(sample)
facts = to_fact_triples(extraction.facts, sample.passage_id)

# Schemas are DERIVED from the typed facts, not emitted by the LLM.
schemas = derive_schema_triples(extraction.facts)
print('Facts:', [f.key() for f in facts])
print('Derived schemas:', [s.key() for s in schemas])

Facts: ['(Our model, trained_for, 3.5 days)', '(Our model, trained_on, eight GPUs)', '(Transformer, generalizes_to, other tasks)', '(Transformer, applied_to, English constituency parsing)', '(Jakob, proposed_replacing, RNNs with self-attention)', '(Jakob, started, evaluation effort)', '(Ashish, designed, first Transformer models)', '(Ashish, implemented, first Transformer models)', '(Ashish, collaborated_with, Illia)', '(Illia, designed, first Transformer models)', '(Illia, implemented, first Transformer models)', '(Noam, proposed, scaled dot-product attention)', '(Noam, proposed, multi-head attention)', '(Noam, proposed, parameter-free position representation)']
Derived schemas: ['(concept, trained_for, date)', '(concept, trained_on, tool)', '(model, generalizes_to, concept)', '(model, applied_to, task)', '(person, proposed_replacing, concept)', '(person, started, concept)', '(person, designed, concept)', '(person, implemented, concept)', '(person, collaborated_with, person)', '(perso

Schemas fall out of entity typing, so they always stay grounded in the facts and passages that produced them.